In [4]:
from langchain_chroma import Chroma
from pdf_chunk import text_spliter
from models import get_model , get_embeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough,RunnableLambda

e:\Rohanta_AI_workbook\adavance_RAG\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
llm= get_model()
embedding_model = get_embeddings()
chunks = text_spliter
chroma_vectorstore=Chroma.from_documents(chunks ,embedding_model )


INFO:sentence_transformers.SentenceTransformer:Use pytorch device_name: cpu
INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2
INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/modules.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/config_sentence_transformers.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/al

In [8]:
chroma_retriever = chroma_vectorstore.as_retriever(search_type="mmr",search_kwargs={"k": 10, "include_metadata": True})

In [6]:
from flashrank import Ranker, RerankRequest

In [7]:
ranker = Ranker(model_name="ms-marco-MiniLM-L-12-v2", cache_dir="/tmp/flashrank")


In [9]:
query = "what is current university of rohanta"
docs = chroma_retriever.invoke(query)

In [10]:
# Step 3: Convert docs to FlashRank format
passages = [
    {"id": i, "text": doc.page_content, "meta": doc.metadata}
    for i, doc in enumerate(docs)
]

In [11]:
# Step 4: Rerank
rerank_request = RerankRequest(query=query, passages=passages)
results = ranker.rerank(rerank_request)

In [12]:
# Step 5: Get top 5
top_k = 5
reranked_docs = [docs[r["id"]] for r in results[:top_k]]

In [15]:
# Step 6: View results
contents = []
for i, r in enumerate(results[:top_k]):
    print(f"Rank {i+1} | Score: {r['score']:.4f}")
    contents.append(r['text'])
    print(f"Text: {r['text'][:100]}...")
    print("---")

Rank 1 | Score: 0.5338
Text: Before transitioning fully into industry, Rohanta taught undergraduate Computer Science to 100+ stud...
---
Rank 2 | Score: 0.1308
Text: reasoning. Rohanta actively experiments with state management, tool routing, execution tracing, and ...
---
Rank 3 | Score: 0.0341
Text: Rohanta uses LangChain to orchestrate LLM workflows. He uses components including PromptTemplate, Ru...
---
Rank 4 | Score: 0.0186
Text: IDENTITY AND CONTACT

Full Name: Rohanta Dinkar Bhamare
Title: AI Engineer
Location: Frankfurt, Germ...
---
Rank 5 | Score: 0.0096
Text: PERSONAL RAG CHATBOT â€” TECHNICAL ARCHITECTURE

This chatbot (rohanta_rag) is built by Rohanta Bham...
---


In [16]:
prompt = ChatPromptTemplate.from_template("""
Answer the question based on the context below.

Context: {context}
Question: {question}
""")

In [17]:
# First join contents into a single string
context_str = "\n\n".join(contents)

# Then pass directly
normal_chain = (
    {"context": RunnableLambda(lambda x: context_str), "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [20]:
query = "what is current university of rohanta"

print(normal_chain.invoke(query))

INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


The context does not mention Rohanta's current or past university. It only mentions that he taught undergraduate Computer Science to 100+ students before transitioning into industry, but it does not provide information about the university where he taught.
